# Benefits Market Intelligence Data Analysis

## Libraries

In [19]:
# Libraries
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
from benefits_market_intelligence.config.paths import (
    DB_PATH,
    FIGURES_PATH,
    ANALYSIS_PATH,
)

## Load gold layer data

In [20]:
# Loading F5500 data
with duckdb.connect(DB_PATH, read_only=True) as con:
    F_5500 = con.sql("SELECT * FROM gold.F_5500").df()

# Loading SCH_A data
with duckdb.connect(DB_PATH, read_only=True) as con:
    SCH_A = con.sql("SELECT * FROM gold.SCH_A").df()

# Loading SCH_C_P1_I2 data
with duckdb.connect(DB_PATH, read_only=True) as con:
    SCH_C_P1_I2 = con.sql("SELECT * FROM gold.SCH_C_P1_I2").df()

## Change object dtypes to str

In [21]:
# Convert pandas-generated "object" datatypes to "str"
F_5500_obj_cols = F_5500.dtypes[F_5500.dtypes == "object"].index
F_5500[F_5500_obj_cols] = F_5500[F_5500_obj_cols].astype("str")

SCH_A_obj_cols = SCH_C_P1_I2.dtypes[SCH_C_P1_I2.dtypes == "object"].index
SCH_C_P1_I2[SCH_A_obj_cols] = SCH_C_P1_I2[SCH_A_obj_cols].astype("str")

## Create paths if missing

In [22]:
# Create paths
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
ANALYSIS_PATH.mkdir(parents=True, exist_ok=True)

# 1. Are broker commissions rising faster than premiums or covered participants?

In [23]:
q1 = (
    SCH_A
    .assign(
        premium=pd.to_numeric(SCH_A["WLFR_TOT_EARNED_PREM_AMT"], errors="coerce"),
        commission=pd.to_numeric(SCH_A["INS_BROKER_COMM_TOT_AMT"], errors="coerce"),
        participants=pd.to_numeric(SCH_A["INS_PRSN_COVERED_EOY_CNT"], errors="coerce"),
        year=pd.to_numeric(SCH_A["FORM_YEAR"], errors="coerce"),
    )
    .query("year >= 2019 and year <= 2024")
    .groupby("year", as_index=False)
    .agg(
        premium=("premium", "sum"),
        commission=("commission", "sum"),
        participants=("participants", "sum"),
    )
    .sort_values("year")
)

# Remove years with no usable denominator and index each measure to the first year.
q1 = q1[q1["premium"] > 0].copy()

for col in ["premium", "commission", "participants"]:
    base = q1.loc[q1["year"].eq(q1["year"].min()), col].iloc[0]
    q1[f"{col}_index"] = q1[col] / base * 100

q1_plot = q1.melt(
    id_vars="year",
    value_vars=["commission_index", "premium_index", "participants_index"],
    var_name="metric",
    value_name="index",
)

q1_plot["metric"] = q1_plot["metric"].map({
    "commission_index": "Broker commissions",
    "premium_index": "Earned premiums",
    "participants_index": "Covered participants",
})

fig = px.line(
    q1_plot,
    x="year",
    y="index",
    color="metric",
    markers=True,
    title="Broker commissions have grown relative to premiums and covered participants",
    labels={
        "year": "Plan year",
        "index": "Indexed value (2019 = 100)",
        "metric": "",
    },
)

fig.update_traces(line=dict(width=3), marker=dict(size=8))
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="",
    margin=dict(l=60, r=30, t=80, b=60),
    title_x=0.02,
    yaxis=dict(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
    ),
    xaxis=dict(dtick=1),
)

fig.show()
fig.write_image(ANALYSIS_PATH / "01-broker-commissions.png", scale = 2)

# 2. Can we predict which plans are likely to terminate, merge, or shrink the following year? (Possible ML question)

In [24]:
plans = (
    F_5500[
        [
            "SPONS_DFE_EIN",
            "SPONS_DFE_PN",
            "FORM_YEAR",
            "TOT_PARTCP_BOY_CNT",
            "TOT_ACTIVE_PARTCP_CNT",
        ]
    ]
    .copy()
)

plans["year"] = pd.to_numeric(plans["FORM_YEAR"], errors="coerce")
plans["participants"] = pd.to_numeric(
    plans["TOT_PARTCP_BOY_CNT"], errors="coerce"
)
plans["active_participants"] = pd.to_numeric(
    plans["TOT_ACTIVE_PARTCP_CNT"], errors="coerce"
)

plans = plans.dropna(
    subset=["SPONS_DFE_EIN", "SPONS_DFE_PN", "year"]
)

plans["plan_id"] = (
    plans["SPONS_DFE_EIN"].astype(str).str.strip()
    + "_"
    + plans["SPONS_DFE_PN"].astype(str).str.strip()
)

# Match each plan to the following year's filing.
prior = plans.rename(
    columns={
        "year": "prior_year",
        "participants": "prior_participants",
        "active_participants": "prior_active",
    }
)

future = plans.rename(
    columns={
        "year": "next_year",
        "participants": "next_participants",
        "active_participants": "next_active",
    }
)

transitions = prior.merge(
    future[
        [
            "plan_id",
            "next_year",
            "next_participants",
            "next_active",
        ]
    ],
    on="plan_id",
    how="left",
)

transitions = transitions[
    transitions["next_year"].eq(transitions["prior_year"] + 1)
].copy()

# Classify what happened to the plan.
transitions["participant_change_pct"] = np.where(
    transitions["prior_participants"] > 0,
    (
        transitions["next_participants"]
        - transitions["prior_participants"]
    )
    / transitions["prior_participants"]
    * 100,
    np.nan,
)

transitions["outcome"] = np.select(
    [
        transitions["next_participants"].isna(),
        transitions["participant_change_pct"] <= -20,
        transitions["participant_change_pct"] >= 20,
    ],
    [
        "No following-year filing",
        "Shrank ≥20%",
        "Grew ≥20%",
    ],
    default="Relatively stable",
)

q2 = (
    transitions
    .groupby(["prior_year", "outcome"], as_index=False)
    .size()
    .rename(columns={"size": "plans"})
)

q2["share"] = (
    q2["plans"]
    / q2.groupby("prior_year")["plans"].transform("sum")
    * 100
)

# Keep the most recent complete transition years.
q2 = q2[q2["prior_year"] <= 2023]

fig = px.density_heatmap(
    q2,
    x="prior_year",
    y="outcome",
    z="share",
    text_auto=".1f",
    title="Observed year-ahead plan outcomes",
    labels={
        "prior_year": "Starting plan year",
        "outcome": "",
        "share": "Share of plans (%)",
    },
    color_continuous_scale="Blues",
)

fig.update_layout(
    template="plotly_white",
    title_x=0.02,
    margin=dict(l=80, r=30, t=80, b=60),
    coloraxis_colorbar=dict(title="Plans (%)"),
)

fig.update_traces(
    hovertemplate=(
        "Starting year: %{x}<br>"
        "Outcome: %{y}<br>"
        "Share: %{z:.1f}%<extra></extra>"
    )
)

fig.show()
fig.write_image(ANALYSIS_PATH / "02-plan-outcomes.png", scale = 2)

# 3. Which brokers are gaining or losing market share?

In [25]:
# Code

# 4. Where are broker commissions unusually high?

In [26]:
q4 = SCH_A.copy()

q4["year"] = pd.to_numeric(q4["FORM_YEAR"], errors="coerce")
q4["premium"] = pd.to_numeric(
    q4["WLFR_TOT_EARNED_PREM_AMT"],
    errors="coerce",
)
q4["commission"] = pd.to_numeric(
    q4["INS_BROKER_COMM_TOT_AMT"],
    errors="coerce",
)

q4 = q4[
    q4["year"].between(2019, 2024)
    & (q4["premium"] > 0)
    & (q4["commission"] >= 0)
    & q4["INS_CARRIER_NAME"].notna()
].copy()

q4["commission_rate"] = (
    q4["commission"] / q4["premium"] * 100
)

# Remove extreme data-entry/reporting outliers for a readable peer comparison.
# The underlying rows are retained; this only limits the visualization.
upper = q4["commission_rate"].quantile(0.99)
q4_plot = q4[q4["commission_rate"] <= upper].copy()

# Focus on carriers with enough observations to make a distribution meaningful.
carrier_counts = (
    q4_plot["INS_CARRIER_NAME"]
    .value_counts()
)

eligible_carriers = carrier_counts[
    carrier_counts >= 50
].index

q4_plot = q4_plot[
    q4_plot["INS_CARRIER_NAME"].isin(eligible_carriers)
]

# Show the 15 carriers with the highest median commission rate.
top_carriers = (
    q4_plot.groupby("INS_CARRIER_NAME")["commission_rate"]
    .median()
    .nlargest(15)
    .index
)

q4_plot = q4_plot[
    q4_plot["INS_CARRIER_NAME"].isin(top_carriers)
]

fig = px.box(
    q4_plot,
    x="commission_rate",
    y="INS_CARRIER_NAME",
    orientation="h",
    points=False,
    title="Broker commission rates by carrier",
    labels={
        "commission_rate": "Broker commission (% of earned premium)",
        "INS_CARRIER_NAME": "",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.02,
    margin=dict(l=260, r=40, t=80, b=60),
    xaxis=dict(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        ticksuffix="%",
        zeroline=False,
    ),
    yaxis=dict(
        autorange="reversed",
    ),
)

fig.update_traces(
    line=dict(width=1.5),
    marker=dict(size=3),
)

fig.show()
fig.write_image(ANALYSIS_PATH / "04-high-broker-commissions.png", scale = 2)

# 5. Think of question

In [27]:
# Code